# OP-09 · Prévision multi-modèle : produits et cartes, brut et calibré

**Notebook opérationnel** — à exécuter à chaque cycle, après OP-08 (calibration et registre).

| | |
|---|---|
| Étape du workflow | E6 (combinaison multi-modèle, Draft §3.4) et E8 (produits) |
| Entrées | cumuls C3S du cycle (prévision et hindcast), registre du cycle `registry/<cycle>/calibration_methods.csv`, synthèses de skill du cycle (poids skill), normales CHIRPS 1991–2020 |
| Sorties | `OUTPUTS_OSF/mme/<cycle>/{raw,calibrated}/` (netCDF et `figures/`), `OUTPUTS_OSF/mme/<cycle>/mme_products_summary.csv`, `OUTPUTS_OSF/climatology/<cycle>/seasonal_share/` |
| Durée indicative | quelques minutes par période et par type (brut, calibré) ; les cartes autant |

Ce que fait cette étape :

1. **chaque modèle produit ses produits** de la période (cumul, anomalie, probabilités des
   terciles et des quintiles, dépassements de la médiane et des seuils en mm, SPI et classes SPI) —
   en **brut**, ou en **calibré** avec la méthode que le registre du cycle lui recommande
   **pour ce produit** ;
2. **le multi-modèle est la moyenne pondérée de ces produits**, et non une combinaison des
   distributions, selon trois schémas : poids **égaux** (la référence, comme `run_forecast_v2`),
   **√(membres)** et **skill** (RPSS médian des terciles en hindcast, ou la discrimination si
   moins de trois modèles ont un RPSS positif) ;
3. **le masque de saison sèche** (critère de l'IRI par défaut) et le masque CEEAC sont appliqués ;
4. **les cartes** suivent les règles d'affichage validées produit par produit (D49) et sont
   nommées `<produit>_<raw|calibrated>_<schéma>_forecast_<période>.png`.

Les décades se combinent en brut sur les sept modèles qui ont un flux journalier (pas NCEP) ;
elles ne sont pas calibrées à ce stade (D42).

## Paramètres

In [ ]:
CYCLE_CONFIG  = "config/cycle_202611.yaml"
VARIABLES     = ["precip"]                      # la température suivra
SCALES        = ["month", "season"]             # "decade" possible en brut seulement
PERIODS       = ["all-months", "all-seasons"]   # None = sous-ensemble de développement ;
                                                # "all", "OND 2026", "season_m1", ...
KINDS         = ["raw", "calibrated"]
SCHEMES       = ["equal", "sqrt_members", "skill"]
DRY_MASK      = None        # None = thresholds.yaml (iri) ; "relative" ou "absolute"
PART_ANNUELLE = True        # produit informatif : part du total annuel normal apportée par la période

In [ ]:
from pathlib import Path
import os
import pandas as pd

REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(REPO)

from eccas_s2s.settings import load_cycle
from eccas_s2s.registry import CALIBRATION, read_register, registry_path

cfg = load_cycle(CYCLE_CONFIG)
print(f"Cycle {cfg.cycle_id} — initialisation {cfg.init_date.date()}")
registre = read_register(cfg, CALIBRATION)
if "calibrated" in KINDS:
    if registre is None:
        raise FileNotFoundError(f"{registry_path(cfg, CALIBRATION)} absent : exécuter OP-08 d'abord")
    print(f"registre du cycle : {len(registre)} décisions, {registre.model.nunique()} modèles, "
          f"échelles {sorted(registre.scale.unique())}")

## 1. Produits multi-modèles

Un netCDF par type, échelle, schéma et produit ; la table `mme_products_summary.csv` garde, pour
chaque combinaison, les modèles entrés, leur méthode et leur poids. Un run partiel **complète**
la table sans la remplacer.

In [ ]:
from eccas_s2s.operations import multimodel

mm = multimodel.run(CYCLE_CONFIG, variables=VARIABLES, scales=SCALES, kinds=KINDS,
                    selection=PERIODS, schemes=SCHEMES, dry_method=DRY_MASK)
print(f"\nStatut : {mm.status} — {len(mm.outputs)} sortie(s)")
for w in mm.warnings:
    print(" ⚠", w)

In [ ]:
synthese = pd.read_csv(cfg.output_root / "mme" / cfg.cycle_id / "mme_products_summary.csv")
display(synthese.groupby(["kind", "scale", "scheme"]).size().unstack("scheme"))

## 2. Cartes de prévision

Une carte par produit, période, type et schéma. Titre sur trois lignes (produit, type et schéma
avec la période en toutes lettres, date d'initialisation) ; en pied, les modèles entrés — avec
leur méthode en calibré et leur poids pour le schéma skill.

In [ ]:
from eccas_s2s.operations import plot_multimodel

cartes = plot_multimodel.run(CYCLE_CONFIG, variables=VARIABLES, scales=SCALES, kinds=KINDS,
                             schemes=SCHEMES)
print(f"Statut : {cartes.status} — {len(cartes.outputs)} carte(s)")

## 3. Produit informatif : part du total annuel

Part du total annuel normal (CHIRPS 1991–2020) apportée par chaque période : elle situe la
période dans le cycle saisonnier de chaque maille, et dit où une anomalie compte vraiment.

In [ ]:
if PART_ANNUELLE:
    from eccas_s2s.operations import season_share

    part = season_share.run(CYCLE_CONFIG, variables=VARIABLES, scales=SCALES, selection=PERIODS)
    print(f"Statut : {part.status} — {len(part.outputs)} sortie(s)")

## 4. Contrôle visuel : terciles de la première saison, brut et calibré, poids égaux

In [ ]:
from IPython.display import Image, display

root = cfg.output_root / "mme" / cfg.cycle_id
for kind in KINDS:
    figures = sorted((root / kind / "figures" / VARIABLES[0] / "season" / "equal" / "terciles")
                     .glob("*.png"))
    if figures:
        print(figures[0].name)
        display(Image(filename=str(figures[0]), width=560))

## 5. Traçabilité

In [ ]:
for c in [mm, cartes] + ([part] if PART_ANNUELLE else []):
    print(f"{c.step:16s} {c.status:8s} {c.run_dir / 'manifest.json'}")